# NeurIPS 2025 - Ariel Data Challenge

This notebook contains my Kaggle solution for the NeurIPS 2025 Ariel Data Challenge. The goal is to recover an exoplanet transmission spectrum from simulated raw observations of the Ariel mission instruments.

The solution follows the full competition workflow: instrument-level preprocessing, transit-aware feature extraction, uncertainty-aware modeling, validation, and generation of the Kaggle submission file.


## 1. Environment and Data Paths

The notebook is designed to run on Kaggle, where the competition dataset is mounted under `/kaggle/input`. Outside Kaggle, local paths can be configured with the `ARIEL_DATA_DIR` and `ARIEL_WORK_DIR` environment variables.


In [1]:
from pathlib import Path
import os

# Detect the execution environment.
ON_KAGGLE = Path("/kaggle").exists()

# Competition dataset root.
KAGGLE_DATASET = "ariel-data-challenge-2025"

if ON_KAGGLE:
    DATA_DIR = Path(f"/kaggle/input/{KAGGLE_DATASET}")
    WORK_DIR = Path("/kaggle/working")
else:
    # Local fallback used outside Kaggle.
    LOCAL_DATA_DIR = Path(os.environ.get("ARIEL_DATA_DIR", "./data"))
    LOCAL_WORK_DIR = Path(os.environ.get("ARIEL_WORK_DIR", "./working"))
    DATA_DIR = LOCAL_DATA_DIR
    WORK_DIR = LOCAL_WORK_DIR

WORK_DIR.mkdir(parents=True, exist_ok=True)

print("ON_KAGGLE =", ON_KAGGLE)
print("DATA_DIR  =", DATA_DIR)
print("WORK_DIR  =", WORK_DIR)

ON_KAGGLE = False
DATA_DIR  = D:\ariel data challange 2025
WORK_DIR  = C:\Users\loulo\OneDrive\Bureau\Kaggle Projects\NeurIPS 2025\working


## 2. Imports and Pipeline Overview

The solution starts from the raw parquet files rather than only using tabular features. Each observation is calibrated, converted into a fixed-length time series, and then passed to a lightweight PyTorch model that predicts both the spectrum `mu` and its uncertainty `sigma`.


In [2]:
import re, math, random, gc
import numpy as np
import pandas as pd

# Machine learning
import pickle, gzip
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [3]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Preprocessing constants
CUT_AIRS_INF, CUT_AIRS_SUP = 39, 321   # Keep 282 AIRS wavelength columns; FGS1 is the broadband channel.
ROI_Y = (10, 22)                       # AIRS rows around the spectral trace.
ROI_FGS = (10, 22, 10, 22)             # 12 x 12 FGS1 aperture.
BIN_AIRS = 30                          # AIRS temporal bin size after CDS.
BIN_FGS  = 30 * 12                     # FGS1 temporal bin size after CDS.

## 3. Metadata and Instrument Timing

The ADC gains and offsets are loaded from the competition metadata. The wavelength grid defines the 283 prediction channels: one broadband FGS1 channel followed by 282 AIRS-CH0 spectral channels.


In [4]:
adc = pd.read_csv(DATA_DIR / "adc_info.csv")
GAIN = {
    "FGS1":     float(adc["FGS1_adc_gain"].iloc[0]),
    "AIRS-CH0": float(adc["AIRS-CH0_adc_gain"].iloc[0]),
}
OFFSET = {
    "FGS1":     float(adc["FGS1_adc_offset"].iloc[0]),
    "AIRS-CH0": float(adc["AIRS-CH0_adc_offset"].iloc[0]),
}

# Wavelength grid: 283 channels = 1 FGS1 + 282 AIRS.
def load_wavelengths(path: Path) -> np.ndarray:
    """Load the 283 prediction wavelengths from wavelengths.csv."""
    row = pd.read_csv(path / "wavelengths.csv", header=0, nrows=1)
    vals = pd.to_numeric(row.iloc[0], errors="coerce").to_numpy(dtype=float)
    if vals.size < 283:
        raise RuntimeError(f"wavelengths.csv must contain 283 value columns; found {vals.size}.")
    return vals[:283]

LAM = load_wavelengths(DATA_DIR)  # (283,)

# Per-frame integration times.
axis_info = pd.read_parquet(DATA_DIR / "axis_info.parquet")

def build_dt_array(sensor: str, n_frames: int) -> np.ndarray:
    """Return the per-frame integration time array for a sensor."""
    sensor = sensor.upper()
    if sensor == "AIRS-CH0":
        if "AIRS-CH0-integration_time" not in axis_info.columns:
            raise RuntimeError("Missing 'AIRS-CH0-integration_time' column in axis_info.parquet")
        dt = axis_info["AIRS-CH0-integration_time"].dropna().values.astype(np.float64)
        if dt.size < n_frames:
            # Tile defensively if metadata is shorter than the signal.
            reps = int(np.ceil(n_frames / dt.size))
            dt = np.tile(dt, reps)[:n_frames]
        elif dt.size > n_frames:
            dt = dt[:n_frames]
        # Match the organizers' convention for odd frames.
        dt = dt.copy()
        dt[1::2] += 0.1
        return dt

    elif sensor == "FGS1":
        # FGS1 has no dedicated axis_info column; use the organizers' cadence convention.
        dt = np.full((n_frames,), 0.1, dtype=np.float64)
        dt[1::2] += 0.1
        return dt

    else:
        raise ValueError(f"Unknown sensor for build_dt_array: {sensor}")

FileNotFoundError: [Errno 2] No such file or directory: 'D:\\ariel data challange 2025\\adc_info.csv'

## 4. Calibration and Signal Preprocessing

The preprocessing stage applies the main detector corrections: ADC conversion, non-linearity correction, dark subtraction, flat-field correction, bad-pixel masking, correlated double sampling, spatial ROI extraction, and temporal binning.


In [ ]:
def adc_convert(arr: np.ndarray, sensor: str) -> np.ndarray:
    arr = arr.astype(np.float64, copy=False)
    return arr / GAIN[sensor] + OFFSET[sensor]

def mask_dead_or_hot(dark: np.ndarray, dead: np.ndarray, k: float = 5.0) -> np.ndarray:
    """Return a bad-pixel mask where True means dead, hot, or invalid."""
    dark = np.asarray(dark, dtype=float)
    dead = np.asarray(dead, dtype=bool)

    mu  = np.nanmean(dark)
    std = np.nanstd(dark)
    if not np.isfinite(std) or std == 0:
        hot = np.zeros_like(dark, dtype=bool)
    else:
        hot = dark > (mu + k * std)

    return dead | hot | ~np.isfinite(dark)

def apply_linear_corr(signal: np.ndarray, coeffs: np.ndarray) -> np.ndarray:
    """Apply the per-pixel non-linearity correction with Horner evaluation."""
    x = signal.astype(np.float64, copy=False)
    out = np.empty_like(x)
    out[...] = coeffs[0]
    for k in range(1, coeffs.shape[0]):
        out *= x
        out += coeffs[k]
    return out

def subtract_dark_per_frame(signal: np.ndarray, dark: np.ndarray, dt_per_frame: np.ndarray) -> None:
    """Subtract dark current in place using one integration time per frame."""
    # Shape for broadcasting over detector pixels.
    dt = dt_per_frame.reshape(-1, 1, 1)
    signal -= dark * dt

def divide_flat_inplace(signal: np.ndarray, flat: np.ndarray, bad_mask: np.ndarray) -> None:
    # Protect invalid flat-field values before in-place division.
    safe_flat = flat.astype(np.float64, copy=False).copy()
    safe_flat[bad_mask | ~np.isfinite(safe_flat) | (safe_flat == 0)] = np.nan
    signal /= safe_flat

def cds_pair_diff(mean_signal: np.ndarray) -> np.ndarray:
    # Pairwise correlated double sampling: odd - even.
    return mean_signal[1::2] - mean_signal[0::2]

def bin_mean(arr: np.ndarray, bin_size: int, ignore_nan: bool = True, allow_partial: bool = True) -> np.ndarray:
    """Average consecutive time bins along the first axis."""
    T = arr.shape[0]
    if bin_size <= 0:
        raise ValueError("bin_size doit être > 0")

    n = T // bin_size
    if n == 0:
        if not allow_partial:
            raise ValueError("bin_size trop grand pour la longueur de la série")
        # Pad to the next full bin when partial bins are allowed.
        n = 1
        pad = n * bin_size - T
        pad_block = np.full((pad, *arr.shape[1:]), np.nan if ignore_nan else 0, dtype=arr.dtype)
        a = np.concatenate([arr, pad_block], axis=0).reshape(n, bin_size, *arr.shape[1:])
    else:
        a = arr[: n * bin_size].reshape(n, bin_size, *arr.shape[1:])

    return np.nanmean(a, axis=1) if ignore_nan else a.mean(axis=1)

## 5. Transit-Aware Observation Features

For each planet observation, the calibrated AIRS and FGS1 signals are aligned into a `(T, 283)` matrix. A simple white-light transit detector provides approximate in-transit and out-of-transit masks used later by the model.


In [5]:
_obs_re = re.compile(r".*_(\d+)\.parquet$")

def _list_obs_indices(pdir: Path, sensor_prefix: str) -> list[int]:
    """Return sorted observation indices available for one sensor."""
    idx = []
    for p in (pdir.glob(f"{sensor_prefix}_signal_*.parquet")):
        m = _obs_re.match(p.name)
        if m:
            idx.append(int(m.group(1)))
    return sorted(set(idx))

# Transit-aware helpers.
DETECT_SLICE = (28, 145)   # Search window in binned time indices.
DELTA_FRAMES = 11          # Margin used to exclude transit ingress/egress.
SMOOTH_WIN   = 21          # Odd boxcar smoothing window.

def _smooth_boxcar(x: np.ndarray, win: int = SMOOTH_WIN) -> np.ndarray:
    """Apply a simple 1D moving average."""
    w = max(3, int(win) | 1)
    if x.size < w:
        return x.astype(np.float64, copy=False)
    k = np.ones(w, dtype=np.float64) / float(w)
    return np.convolve(x.astype(np.float64, copy=False), k, mode="same")

def detect_transit_p1p2(white_air: np.ndarray,
                        search: tuple[int,int] = DETECT_SLICE) -> tuple[int,int]:
    """Approximate transit boundaries on the smoothed AIRS white-light curve."""
    s = _smooth_boxcar(white_air, SMOOTH_WIN)
    a, b = int(search[0]), int(search[1])
    b = min(b, len(s)-1)
    midx = int(np.argmin(s[a:b])) + a

    s1 = s[:midx]; s2 = s[midx:]
    if s1.size < 3 or s2.size < 3:
        return 0, len(s)-1

    g1 = np.gradient(s1); g2 = np.gradient(s2)
    g1_max = np.max(np.abs(g1)) if g1.size else 0.0
    g2_max = np.max(np.abs(g2)) if g2.size else 0.0
    if g1_max > 0: g1 /= g1_max
    if g2_max > 0: g2 /= g2_max

    p1 = int(np.argmin(g1))
    p2 = int(np.argmax(g2)) + midx
    p1 = max(0, min(p1, len(s)-1))
    p2 = max(0, min(p2, len(s)-1))
    if p2 <= p1:
        p1, p2 = sorted((p1, p2))
    return p1, p2

def _crop_offset(T_src: int, T_tgt: int) -> int:
    """Return the centered crop offset used by pad_or_crop_time."""
    return 0 if (T_src <= T_tgt) else (T_src - T_tgt) // 2

def build_masks_after_crop(T_src: int, T_tgt: int, p1: int, p2: int,
                           delta: int = DELTA_FRAMES) -> tuple[np.ndarray, np.ndarray]:
    """Build in-transit and out-of-transit masks after centered cropping."""
    start = _crop_offset(T_src, T_tgt)
    p1c, p2c = p1 - start, p2 - start
    p1c = max(0, min(p1c, T_tgt-1))
    p2c = max(0, min(p2c, T_tgt-1))
    if p2c < p1c:
        p1c, p2c = p2c, p1c

    mask_in  = np.zeros((T_tgt,), dtype=bool)
    mask_oot = np.zeros((T_tgt,), dtype=bool)

    lo_in = max(0, p1c + delta)
    hi_in = max(0, p2c - delta)
    if hi_in > lo_in:
        mask_in[lo_in:hi_in] = True

    lo_oot1 = 0
    hi_oot1 = max(0, p1c - delta)
    if hi_oot1 > lo_oot1:
        mask_oot[lo_oot1:hi_oot1] = True

    lo_oot2 = min(T_tgt, p2c + delta)
    hi_oot2 = T_tgt
    if hi_oot2 > lo_oot2:
        mask_oot[lo_oot2:hi_oot2] = True

    # If detection fails, keep all valid frames available for the OOT baseline.
    if not mask_in.any():
        mask_oot[:] = True
    return mask_in, mask_oot

def preprocess_planet_obs(root: Path, split: str, pid: int) -> dict[int, np.ndarray]:
    """Preprocess all paired AIRS and FGS1 observations for one planet.

    Returns a dictionary keyed by observation index. Each value contains:
    - M: calibrated and binned signal with shape (T, 283)
    - p1, p2: approximate transit boundary indices in binned time
    """
    pdir = root / split / str(pid)

    # Keep only observations available for both sensors.
    obs_air = _list_obs_indices(pdir, "AIRS-CH0")
    obs_fgs = _list_obs_indices(pdir, "FGS1")
    obs_common = sorted(set(obs_air).intersection(obs_fgs))
    if not obs_common:
        return {}

    results: dict[int, np.ndarray] = {}

    for obs in obs_common:
        # AIRS-CH0 preprocessing.
        df = pd.read_parquet(pdir / f"AIRS-CH0_signal_{obs}.parquet")
        sig = df.values.reshape(11250, 32, 356)
        sig = adc_convert(sig, "AIRS-CH0")
        
        dark = pd.read_parquet(pdir / f"AIRS-CH0_calibration_{obs}/dark.parquet").to_numpy().reshape(32, 356)
        dead = (
            pd.read_parquet(pdir / f"AIRS-CH0_calibration_{obs}/dead.parquet")
            .to_numpy()
            .astype(bool)
            .reshape(32, 356)
        )
        flat = pd.read_parquet(pdir / f"AIRS-CH0_calibration_{obs}/flat.parquet").to_numpy().reshape(32, 356)
        lin = (
            pd.read_parquet(pdir / f"AIRS-CH0_calibration_{obs}/linear_corr.parquet")
            .to_numpy()
            .astype(np.float64)
            .reshape(6, 32, 356)
        )
        
        # Keep the 282 AIRS wavelength columns used for prediction.
        sig  = sig[:, :, CUT_AIRS_INF:CUT_AIRS_SUP]
        dark = dark[:, CUT_AIRS_INF:CUT_AIRS_SUP]
        dead = dead[:, CUT_AIRS_INF:CUT_AIRS_SUP]
        flat = flat[:, CUT_AIRS_INF:CUT_AIRS_SUP]
        lin  = lin[:, :, CUT_AIRS_INF:CUT_AIRS_SUP]
        
        # Detector corrections and spatial extraction.
        bad = mask_dead_or_hot(dark, dead, k=5.0)
        y0, y1 = ROI_Y
        sig[:, y0:y1, :] = apply_linear_corr(sig[:, y0:y1, :], np.flip(lin[:, y0:y1, :], axis=0))
        dt_airs = build_dt_array("AIRS-CH0", sig.shape[0])  # (11250,)
        subtract_dark_per_frame(sig, dark, dt_airs)
        divide_flat_inplace(sig[:, y0:y1, :], flat[y0:y1, :], bad[y0:y1, :])
        airs_series = np.nanmean(sig[:, y0:y1, :], axis=1)       # (T_full, 282)
        
        # CDS and temporal binning.
        try:
            airs_cds = cds_pair_diff(airs_series)                # (T/2, 282)
            if airs_cds.shape[0] == 0:
                raise ValueError("AIRS_cds vide")
            airs_bin = bin_mean(airs_cds, BIN_AIRS, allow_partial=True)  # (T_AIRS, 282)
        except Exception:
            continue
        
        del df, sig, dark, dead, flat, lin, bad, airs_series, airs_cds
        gc.collect()
        
        # FGS1 preprocessing.
        df = pd.read_parquet(pdir / f"FGS1_signal_{obs}.parquet")
        sig = df.values.reshape(135000, 32, 32)
        sig = adc_convert(sig, "FGS1")
        
        dark = pd.read_parquet(pdir / f"FGS1_calibration_{obs}/dark.parquet").to_numpy().reshape(32, 32)
        dead = pd.read_parquet(pdir / f"FGS1_calibration_{obs}/dead.parquet").to_numpy().astype(bool).reshape(32, 32)
        flat = pd.read_parquet(pdir / f"FGS1_calibration_{obs}/flat.parquet").to_numpy().reshape(32, 32)
        lin = (
            pd.read_parquet(pdir / f"FGS1_calibration_{obs}/linear_corr.parquet")
            .to_numpy()
            .astype(np.float64)
            .reshape(6, 32, 32)
        )
        
        y0, y1, x0, x1 = ROI_FGS
        sig  = sig[:, y0:y1, x0:x1]
        dark = dark[y0:y1, x0:x1]
        dead = dead[y0:y1, x0:x1]
        flat = flat[y0:y1, x0:x1]
        lin  = lin[:, y0:y1, x0:x1]
        
        bad = mask_dead_or_hot(dark, dead, k=5.0)
        sig = apply_linear_corr(sig, np.flip(lin, axis=0))
        dt_fgs = build_dt_array("FGS1", sig.shape[0])  # (135000,)
        subtract_dark_per_frame(sig, dark, dt_fgs)
        divide_flat_inplace(sig, flat, bad)
        
        fgs_white = np.nanmean(sig, axis=(1, 2))                 # (T_full,)
        try:
            fgs_cds = cds_pair_diff(fgs_white[:, None])[:, 0]    # (T/2,)
            if fgs_cds.shape[0] == 0:
                raise ValueError("FGS_cds vide")
            fgs_bin = bin_mean(fgs_cds[:, None], BIN_FGS, allow_partial=True)[:, 0]  # (T_FGS,)
        except Exception:
            continue
        
        del df, sig, dark, dead, flat, lin, bad, fgs_white, fgs_cds
        gc.collect()
        
        # Align both sensors into one (T, 283) matrix.
        T = int(min(airs_bin.shape[0], fgs_bin.shape[0]))
        if T <= 0:
            continue
        mat = np.concatenate([fgs_bin[:T, None], airs_bin[:T, :]], axis=1)   # (T, 283)
        
        # Estimate transit boundaries from the binned AIRS white-light curve.
        white_air = np.nanmean(airs_bin[:T, :], axis=1)
        p1, p2 = detect_transit_p1p2(white_air, DETECT_SLICE)
        
        results[obs] = {
            "M":  mat.astype(np.float32, copy=False),
            "p1": int(p1),
            "p2": int(p2),
        }
        
    return results

## 6. Training Configuration

The model is trained on observation-level samples with a fixed temporal length. Validation is grouped by `planet_id`, so observations from the same planet do not leak across train and validation splits.


In [6]:
SEED          = 42
DEVICE        = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SPLIT_TRAIN   = "train"

# Fixed temporal length after CDS and binning.
T_TARGET      = 187

# Model hyperparameters.
LR            = 1e-3
WD            = 1e-4
EPOCHS        = 75
BATCH_SIZE    = 16

# Spectral smoothness penalty on second differences of mu.
SMOOTH_LMBDA  = 1e-3

# Channel weights.
W_FGS_RAW     = 57.846    # FGS1 receives a much larger metric weight.

# Minimum uncertainty floors.
SIGMA_MIN_FGS  = 1e-4
SIGMA_MIN_AIRS = 1e-4

# Cache directory.
CACHE_DIR     = WORK_DIR
CACHE_DIR.mkdir(parents=True, exist_ok=True)

In [7]:
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_seed()

def planet_features16(star_row: np.ndarray) -> np.ndarray:
    """Create 16 metadata features from 8 raw stellar/planet parameters."""
    x = star_row.astype(np.float32)
    return np.concatenate([x, x**2], axis=0).astype(np.float32)

def standardize_fit(X: np.ndarray):
    mu = X.mean(axis=0)
    sd = X.std(axis=0)
    sd = np.where(sd < 1e-8, 1.0, sd)
    return mu.astype(np.float32), sd.astype(np.float32)

def standardize_apply(X: np.ndarray, mu: np.ndarray, sd: np.ndarray):
    return ((X - mu) / sd).astype(np.float32)

def pad_or_crop_time(M: np.ndarray, T_target: int):
    """Center-crop or zero-pad an observation to a fixed temporal length."""
    T, C = M.shape
    mask = np.zeros((T_target,), dtype=bool)
    if T == T_target:
        mask[:] = True
        return M.astype(np.float32), mask
    if T > T_target:
        start = (T - T_target) // 2
        end   = start + T_target
        out = M[start:end]
        mask[:] = True
        return out.astype(np.float32), mask
    out = np.zeros((T_target, C), dtype=np.float32)
    out[:T] = M.astype(np.float32)
    mask[:T] = True
    return out, mask



## 7. Observation Dataset Construction

Each sample contains the calibrated spectral time series, validity masks, in-transit and out-of-transit masks, out-of-transit statistics, stellar/planetary metadata features, and the 283-dimensional target spectrum.


In [8]:
class ObsDataset(Dataset):
    """Observation-level PyTorch dataset."""
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        s = self.samples[i]
        return (
            torch.from_numpy(s["X_spec"]),
            torch.from_numpy(s["time_mask"]),
            torch.from_numpy(s["mask_in"]),
            torch.from_numpy(s["mask_oot"]),
            torch.from_numpy(s["oot_mean"]),
            torch.from_numpy(s["oot_std"]),
            torch.from_numpy(s["X_feat"]),
            torch.from_numpy(s["y"]),
            s["meta"],
        )

# Parallel observation-list builder.

from concurrent.futures import ThreadPoolExecutor, as_completed

# Worker-local state initialized once per process.
_GLOBALS = {
    "DATA_DIR": None,
    "SPLIT": None,
    "T_TARGET": None,
    "FEAT16": None,
    "YTRUE": None,
    "HAS_Y": False,
}
def _init_worker(data_dir, split, t_target, feat16_table, ytrue_table, has_y):
    _GLOBALS["DATA_DIR"]  = Path(data_dir)
    _GLOBALS["SPLIT"]     = split
    _GLOBALS["T_TARGET"]  = int(t_target)
    _GLOBALS["FEAT16"]    = feat16_table
    _GLOBALS["YTRUE"]     = ytrue_table
    _GLOBALS["HAS_Y"]     = bool(has_y)


def _process_one_pid(pid: int):
    """Process all observations for one planet.

    Returns observation-level samples and one per-channel temporal
    variance vector per usable observation.
    """
    DATA_DIR  = _GLOBALS["DATA_DIR"]
    SPLIT     = _GLOBALS["SPLIT"]
    T_TARGET  = _GLOBALS["T_TARGET"]
    FEAT16    = _GLOBALS["FEAT16"]
    YTRUE     = _GLOBALS["YTRUE"]
    HAS_Y     = _GLOBALS["HAS_Y"]

    out_samples = []
    out_vars    = []

    # One calibrated (T, 283) matrix per usable observation.
    obs_dict = preprocess_planet_obs(DATA_DIR, SPLIT, pid)
    if not obs_dict:
        return out_samples, out_vars

    # Metadata features for this planet.
    feat16 = FEAT16[int(pid)]

    # Target spectrum is available only for the train split.
    y_vec = (YTRUE[int(pid)] if HAS_Y else np.zeros((283,), dtype=np.float32)).astype(np.float32)

    for obs_idx, pack in sorted(obs_dict.items()):
        M  = pack["M"]         # (T_src, 283)
        p1 = int(pack["p1"]); p2 = int(pack["p2"])

        # Fixed-length time axis and validity mask.
        X_spec, time_mask = pad_or_crop_time(M, T_TARGET)             # (T_tgt,283), (T_tgt,)
        # Transit masks after the same crop.
        mask_in, mask_oot = build_masks_after_crop(M.shape[0], T_TARGET, p1, p2, DELTA_FRAMES)

        # Per-channel out-of-transit baseline.
        valid_oot = (mask_oot & time_mask)
        if valid_oot.any():
            oot_mean = np.nanmean(X_spec[valid_oot], axis=0)
            oot_std  = np.nanstd( X_spec[valid_oot], axis=0, ddof=0)
        else:
            valid = time_mask
            oot_mean = np.nanmean(X_spec[valid], axis=0)
            oot_std  = np.nanstd( X_spec[valid], axis=0, ddof=0)
        
        # Replace invalid values and keep a strictly positive scale.
        oot_mean = np.nan_to_num(oot_mean, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
        oot_std  = np.nan_to_num(oot_std,  nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
        oot_std  = np.maximum(oot_std, 1e-6)

        # Per-channel temporal variance retained as a diagnostic.
        var_lambda = np.nanvar(X_spec[time_mask], axis=0, ddof=0).astype(np.float64)
        var_lambda = np.nan_to_num(var_lambda, nan=0.0, posinf=0.0, neginf=0.0)

        out_samples.append({
            "X_spec": X_spec.astype(np.float32),
            "time_mask": time_mask.astype(bool),
            "mask_in":  mask_in.astype(bool),
            "mask_oot": mask_oot.astype(bool),
            "oot_mean": oot_mean.astype(np.float32),
            "oot_std":  np.maximum(oot_std, 1e-12).astype(np.float32),
            "X_feat":   feat16.astype(np.float32),
            "y":        y_vec,
            "meta": {"planet_id": int(pid), "obs_idx": int(obs_idx)},
        })
        out_vars.append(var_lambda)

    return out_samples, out_vars


def build_observation_list_parallel(
    data_dir: Path,
    split: str,
    progress_every: int = 20,
    max_workers: int | None = None,
):
    """Build observation-level samples in parallel.

    Returns samples, feature standardization statistics, median per-channel
    temporal variance, and planet_id groups for grouped validation.
    """
    data_dir = Path(data_dir)
    is_train = (split == "train")

    # Lightweight tables kept in the main process.
    star_df = pd.read_csv(data_dir / f"{split}_star_info.csv").set_index("planet_id").sort_index()
    pids = star_df.index.astype(np.int64).tolist()

    # Precompute the small metadata-feature table: {pid: (16,)}.
    feat16_table = {}
    for pid in pids:
        vals8 = star_df.loc[pid].values.astype(np.float32)  # (8,)
        feat16_table[int(pid)] = planet_features16(vals8)    # (16,)

    # Train labels, when available.
    ytrue_table = None
    if is_train:
        train_truth = pd.read_csv(data_dir / "train.csv").set_index("planet_id").sort_index()
        # Keep only planets present in both metadata and target tables.
        pids = sorted(set(pids).intersection(set(train_truth.index.astype(np.int64).tolist())))
        ytrue_table = {int(pid): train_truth.loc[pid].values.astype(np.float32) for pid in pids}

    n_planets = len(pids)
    if n_planets == 0:
        raise RuntimeError(f"No planets found in {split}.")

    # Process pool.
    if max_workers is None:
        # Use a conservative worker count to avoid saturating disk I/O.
        max_workers = max(1, os.cpu_count() // 2)

    print(f"[build_observation_list_parallel] planets={n_planets} | workers={max_workers}")

    samples_all = []
    var_all     = []

    # Initialize worker-local state.
    with ThreadPoolExecutor(
        max_workers=max_workers,
        initializer=_init_worker,
        initargs=(str(data_dir), split, T_TARGET, feat16_table, ytrue_table, is_train),
    ) as ex:
        futs = {ex.submit(_process_one_pid, int(pid)): int(pid) for pid in pids}

        skipped = 0
        for i, fut in enumerate(as_completed(futs), start=1):
            pid = futs[fut]
            try:
                s_list, v_list = fut.result()
                if s_list:
                    samples_all.extend(s_list)
                    var_all.extend(v_list)
                else:
                    skipped += 1
            except Exception as e:
                skipped += 1
                if skipped <= 5:
                    print(f"[WARN] pid={pid} skipped: {type(e).__name__}: {e}")

            # Progress logging.
            if (i % progress_every) == 0 or i == n_planets:
                print(f"  {i:>4}/{n_planets:<4}  pid={pid}  (+{len(samples_all)} observations)")

    if len(samples_all) == 0:
        raise RuntimeError("No observations were built.")

    # Standardize metadata features after sample collection.
    feat_mat = np.stack([s["X_feat"] for s in samples_all], axis=0)  # (N_obs,16)
    mu, sd   = standardize_fit(feat_mat)
    for s in samples_all:
        s["X_feat"] = standardize_apply(s["X_feat"], mu, sd)

    # Median temporal variance per channel.
    var_all = np.vstack(var_all)  # (N_obs, 283)
    med_var = np.nanmedian(var_all, axis=0).astype(np.float64)

    # Group labels for planet-level validation splits.
    groups = np.array([s["meta"]["planet_id"] for s in samples_all], dtype=np.int64)

    print(f"[build_observation_list_parallel] Observations: {len(samples_all)} | med_var.shape={med_var.shape}")
    return samples_all, (mu, sd), med_var, groups

## 8. Uncertainty Floor and Channel Weights

The competition score rewards calibrated uncertainty, not just accurate means. This section defines a per-channel uncertainty floor and emphasizes the heavily weighted FGS1 channel.


In [9]:
def make_sigma_floor_from_labels(std_y: np.ndarray, frac: float = 0.25) -> np.ndarray:
    """Build a per-channel lower bound for predicted uncertainty."""
    std_y = np.asarray(std_y, dtype=np.float64)
    sigma = np.maximum(frac * std_y, 1e-12).astype(np.float32)
    # Separate lower bounds for FGS1 and AIRS channels.
    sigma[0]  = max(SIGMA_MIN_FGS,  float(sigma[0]))
    sigma[1:] = np.maximum(SIGMA_MIN_AIRS, sigma[1:])
    return sigma

# Channel weights used in the internal weighted NLL.
def make_channel_weights() -> np.ndarray:
    w = np.ones(283, dtype=np.float32)
    w[0] = W_FGS_RAW
    return w

def split_train_val(samples, groups, val_frac=0.1, seed=SEED):
    """Create a grouped train/validation split by planet_id."""
    rng = np.random.default_rng(seed)
    uniq = np.array(sorted(set(groups.tolist())))
    n_val = max(1, int(len(uniq) * val_frac))
    val_planets = set(rng.choice(uniq, size=n_val, replace=False).tolist())
    idx_tr, idx_va = [], []
    for i, g in enumerate(groups):
        (idx_va if g in val_planets else idx_tr).append(i)
    return idx_tr, idx_va

## 9. Model and Loss

The model combines per-channel time-series statistics, in-transit/out-of-transit summaries, physical metadata features, and a learned wavelength embedding. It outputs `mu` and `sigma` and is optimized with a weighted Gaussian negative log-likelihood plus a small spectral smoothness penalty.


In [10]:
def masked_mean_std(X_spec, time_mask, eps=1e-8):
    """Compute per-channel mean and std over valid time steps."""
    m = time_mask.float().unsqueeze(-1)
    denom = m.sum(dim=1).clamp_min(1.0)
    mean_t = (X_spec * m).sum(dim=1) / denom
    ex2    = ((X_spec ** 2) * m).sum(dim=1) / denom
    var_t  = torch.relu(ex2 - mean_t**2)
    std_t  = torch.sqrt(var_t + eps)
    return mean_t, std_t

def masked_mean_std_with_mask(X_spec, base_mask, extra_mask, eps=1e-8):
    """Compute per-channel statistics using the intersection of two masks."""
    m = (base_mask & extra_mask).float().unsqueeze(-1)
    denom = m.sum(dim=1).clamp_min(1.0)
    mean_t = (X_spec * m).sum(dim=1) / denom
    ex2    = ((X_spec ** 2) * m).sum(dim=1) / denom
    var_t  = torch.relu(ex2 - mean_t**2)
    std_t  = torch.sqrt(var_t + eps)
    return mean_t, std_t

class SpecMixMLP(nn.Module):
    """MLP-based predictor for spectrum means and calibrated uncertainties.

    Inputs combine binned spectral time series, transit masks, OOT baseline
    statistics, metadata features, and a per-channel uncertainty floor.
    """
    def __init__(self, L=283, feat_dim=16, emb_dim=16, mlp_hidden=128,
                 c=48, dropout=0.10, y_mean=None, y_std=None):
        super().__init__()
        self.L = L
        self.feat_dim = feat_dim
        self.emb = nn.Embedding(L, emb_dim)
        self.reg_idx = torch.arange(L, dtype=torch.long)

        # Descriptor size: global stats, OOT stats, IN stats, naive depth, metadata, embedding.
        d_in = 7 + feat_dim + emb_dim
        self.mlp = nn.Sequential(
            nn.Linear(d_in, mlp_hidden), nn.GELU(), nn.LayerNorm(mlp_hidden), nn.Dropout(dropout),
            nn.Linear(mlp_hidden, c),    nn.GELU(), nn.LayerNorm(c),          nn.Dropout(dropout),
        )

        self.mu_head = nn.Linear(c, 1)
        self.sg_head = nn.Linear(c, 1)
        self.softplus = nn.Softplus(beta=1.0)

        # Label statistics follow the model device as buffers.
        assert y_mean is not None and y_std is not None, "y_mean and y_std must be provided"
        self.register_buffer("y_mean", torch.as_tensor(y_mean, dtype=torch.float32))
        self.register_buffer("y_std",  torch.as_tensor(np.maximum(y_std, 1e-8), dtype=torch.float32))

        # Softplus temperature used to keep mu positive.
        tau_val = float(np.median(np.asarray(y_std, dtype=np.float32)))
        self.register_buffer("mu_tau", torch.as_tensor(max(tau_val, 1e-8), dtype=torch.float32))

    def forward(self, X_spec, time_mask, mask_in, mask_oot, oot_mean, oot_std, X_feat, sigma_floor_phys):
        B, T, L = X_spec.shape
        assert L == self.L

        # Global valid-frame statistics.
        mean_t, std_t = masked_mean_std(X_spec, time_mask)  # (B,L)

        # In-transit statistics.
        mean_in, std_in = masked_mean_std_with_mask(X_spec, time_mask, mask_in)   # (B,L)

        # Out-of-transit baseline from the dataset.
        oot_mean = oot_mean.to(X_spec.device)     # (B,L)
        oot_std  = oot_std.to (X_spec.device)     # (B,L)

        # Naive non-negative transit depth proxy.
        depth_naif = torch.relu(oot_mean - mean_in)  # (B,L)

        # Per-channel descriptor assembly.
        feat = X_feat.unsqueeze(1).expand(B, L, self.feat_dim)   # (B,L,feat_dim)
        emb  = self.emb(self.reg_idx.to(X_spec.device)).unsqueeze(0).expand(B, L, -1)

        desc = torch.cat([
            mean_t.unsqueeze(-1), std_t.unsqueeze(-1),
            oot_mean.unsqueeze(-1), oot_std.unsqueeze(-1),
            mean_in.unsqueeze(-1), std_in.unsqueeze(-1),
            depth_naif.unsqueeze(-1),
            feat, emb
        ], dim=-1)  # (B,L, 7+feat+emb)

        z = self.mlp(desc)   # (B,L,C)

        # Heads and reprojection to physical scale.
        mu_norm   = self.mu_head(z).squeeze(-1)
        sg_raw    = self.sg_head(z).squeeze(-1)
        sigma_floor_norm = torch.clamp(sigma_floor_phys / self.y_std, min=1e-8)
        sigma_norm = self.softplus(sg_raw) + sigma_floor_norm
        mu_phys    = self.y_mean + self.y_std * mu_norm
        sigma_phys = self.y_std * sigma_norm

        tau = self.mu_tau
        mu_phys_pos = tau * torch.nn.functional.softplus(mu_phys / tau)
        return mu_phys_pos, sigma_phys

def nll_weighted(mu, sigma, y_true, chan_w, smooth_lambda=SMOOTH_LMBDA):
    """Weighted Gaussian NLL with a small spectral smoothness penalty."""
    eps   = 1e-8
    sigma = torch.clamp(sigma, min=eps)
    z     = (y_true - mu) / sigma
    per   = 0.5 * (math.log(2*math.pi) + 2.0*torch.log(sigma) + z**2)  # (B,L)
    per   = per * chan_w.unsqueeze(0)
    base  = per.mean()

    if smooth_lambda > 0:
        d2  = mu[:, 2:] - 2.0*mu[:, 1:-1] + mu[:, :-2]
        sm  = (d2**2).mean()
        return base + smooth_lambda * sm
    return base

## 10. Preprocessing Cache

Raw preprocessing is expensive because each planet requires reading and calibrating large AIRS and FGS1 parquet files. The processed observation list is cached so training can be resumed without repeating detector-level preprocessing.


In [ ]:
# Build and cache train observations.
print("Building train observations...")
train_samples, (feat_mu, feat_sd), med_var_train, groups = build_observation_list_parallel(
    DATA_DIR, SPLIT_TRAIN, progress_every=1, max_workers=2
)


OUT_DIR = WORK_DIR
OUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_PATH = OUT_DIR / "train_obs_cache.pkl.gz"

payload = {
    "train_samples": train_samples,
    "feat_mu": feat_mu,               # (16,)
    "feat_sd": feat_sd,               # (16,)
    "med_var_train": med_var_train,   # (283,)
    "groups": groups,                 # (N_obs,)
}

with gzip.open(CACHE_PATH, "wb") as f:
    pickle.dump(payload, f, protocol=pickle.HIGHEST_PROTOCOL)

print(f"Cache written to: {CACHE_PATH}")

Building train observations...
[build_observation_list_parallel] planets=1100 | workers=2
     1/1100  pid=34983  (+1 observations)
     2/1100  pid=1873185  (+3 observations)
     3/1100  pid=3849793  (+4 observations)


## 11. Reload Cached Observations

The following cell reloads the cached observation-level dataset, feature normalization statistics, per-channel variance diagnostics, and planet grouping information.


In [12]:
CACHE_PATH = Path(WORK_DIR / "train_obs_cache.pkl.gz")

with gzip.open(CACHE_PATH, "rb") as f:
    cache = pickle.load(f)

train_samples  = cache["train_samples"]
feat_mu        = cache["feat_mu"]
feat_sd        = cache["feat_sd"]
med_var_train  = cache["med_var_train"]
groups         = cache["groups"]

print(
    "Cache loaded:",
    len(train_samples), "observations | med_var_train:", med_var_train.shape,
    "| feat_mu/sd:", feat_mu.shape, feat_sd.shape,
    "| groups:", groups.shape,
)


Cache loaded: 1209 observations | med_var_train: (283,) | feat_mu/sd: (16,) (16,) | groups: (1209,)


In [13]:
# Grouped split by planet_id.
idx_tr, idx_va = split_train_val(train_samples, groups, val_frac=0.1, seed=SEED)
train_ds = ObsDataset([train_samples[i] for i in idx_tr])
val_ds   = ObsDataset([train_samples[i] for i in idx_va])

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  drop_last=False)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

print(f"TRAIN observations: {len(train_ds)} | VAL observations: {len(val_ds)}")


TRAIN observations: 1094 | VAL observations: 115


## 12. Training

The model is trained with AdamW, a warmup-cosine learning-rate schedule, gradient clipping, and best-validation checkpointing in memory. The validation metric shown here is the internal weighted NLL used for model selection.

The validation split is used as an internal training monitor. Label normalization and uncertainty floors are estimated from all training samples, following the final Kaggle training setup, so the displayed validation NLL should be read as a sanity check rather than a strict cross-validation estimate.


In [14]:
DROPOUT = 0.10

# Label statistics for normalization and physical-scale reprojection.
Y_mat   = np.stack([s["y"] for s in train_samples], axis=0)   # (N_obs, 283)
y_mean  = Y_mat.mean(axis=0).astype(np.float32)               # (283,)
y_std   = Y_mat.std(axis=0, ddof=1).astype(np.float32)        # (283,)
y_std[y_std < 1e-8] = 1.0

# Final sigma_floor used by the model.
sigma_floor_np = make_sigma_floor_from_labels(y_std, frac=0.25)         # (283,)
chan_weights_np = make_channel_weights()                   # (283,)
SIG_FLOOR = torch.from_numpy(sigma_floor_np).to(DEVICE)
W_CHAN    = torch.from_numpy(chan_weights_np).to(DEVICE)

model = SpecMixMLP(
    L=283, feat_dim=16, emb_dim=16, mlp_hidden=128,
    c=48, dropout=DROPOUT,
    y_mean=y_mean, y_std=y_std,
).to(DEVICE)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)

class WarmupCosine:
    def __init__(self, optim, base_lr, warmup_epochs, total_epochs):
        self.optim = optim
        self.base_lr = base_lr
        self.warmup = warmup_epochs
        self.total = total_epochs
        self.step_idx = 0

    def step(self, epoch):
        self.step_idx = epoch
        if epoch <= self.warmup:
            lr = self.base_lr * epoch / max(1, self.warmup)
        else:
            t = (epoch - self.warmup) / max(1, self.total - self.warmup)
            lr = 0.5 * self.base_lr * (1 + math.cos(math.pi * t))
        for g in self.optim.param_groups:
            g['lr'] = lr

WARMUP_EPOCHS = 3
scheduler = WarmupCosine(optimizer, base_lr=LR, warmup_epochs=WARMUP_EPOCHS, total_epochs=EPOCHS)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_params:,}")

def run_epoch(loader, training=True):
    (model.train() if training else model.eval())
    total = 0.0; n = 0

    # Disable gradients during validation.
    grad_ctx = torch.enable_grad() if training else torch.no_grad()

    with grad_ctx:
        for X_spec, time_mask, mask_in, mask_oot, oot_mean, oot_std, X_feat, y, _meta in loader:
            X_spec    = X_spec.to(DEVICE)
            time_mask = time_mask.to(DEVICE)
            mask_in   = mask_in.to(DEVICE)
            mask_oot  = mask_oot.to(DEVICE)
            oot_mean  = oot_mean.to(DEVICE)
            oot_std   = oot_std.to(DEVICE)
            X_feat    = X_feat.to(DEVICE)
            y         = y.to(DEVICE)

            if training:
                optimizer.zero_grad(set_to_none=True)

            mu, sigma = model(
                X_spec, time_mask, mask_in, mask_oot, oot_mean, oot_std, X_feat, SIG_FLOOR
            )
            loss = nll_weighted(mu, sigma, y, W_CHAN, SMOOTH_LMBDA)

            if training:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                optimizer.step()

            bs = X_spec.size(0)
            total += loss.detach().item() * bs
            n += bs

    return total / max(1, n)

best_val, best_state = float("inf"), None
for epoch in range(1, EPOCHS+1):
    scheduler.step(epoch)
    tr = run_epoch(train_loader, training=True)
    va = run_epoch(val_loader,   training=False)
    
    if va < best_val:
        best_val = va
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    if (epoch == 1) or (epoch % 5 == 0) or (epoch == EPOCHS):
        print(f"[{epoch:02d}/{EPOCHS}] train NLL={tr:.5f} | val NLL={va:.5f} (best {best_val:.5f})")

if best_state is not None:
    model.load_state_dict(best_state)

Trainable parameters: 16,290


KeyError: 'mask_in'

## 13. Test Inference and Submission

Test observations are preprocessed with the same pipeline as training. When multiple observations exist for one planet, predictions are aggregated with inverse-variance weighting before writing `submission.csv`.


In [ ]:
# Aggregate multiple observations with inverse-variance weighting.
def _aggregate_observations(mu_list, sigma_list, eps=1e-12):
    mu_arr    = np.stack(mu_list, axis=0)
    sigma_arr = np.clip(np.stack(sigma_list, axis=0), eps, None)
    w = 1.0 / (sigma_arr ** 2)
    w_sum = np.sum(w, axis=0)
    mu_agg  = np.sum(w * mu_arr, axis=0) / np.clip(w_sum, eps, None)
    sig_agg = 1.0 / np.sqrt(np.clip(w_sum, eps, None))
    return mu_agg.astype(np.float32), sig_agg.astype(np.float32)

def infer_one_observation_specmix(
    X_spec_np: np.ndarray,
    time_mask_np: np.ndarray,
    mask_in_np: np.ndarray,
    mask_oot_np: np.ndarray,
    oot_mean_np: np.ndarray,
    oot_std_np: np.ndarray,
    X_feat_np: np.ndarray,
):
    """Run the trained model on one preprocessed observation."""
    model.eval()
    with torch.no_grad():
        X_spec    = torch.from_numpy(X_spec_np[None, ...]).to(DEVICE)      # (1,T,L)
        time_mask = torch.from_numpy(time_mask_np[None, ...]).to(DEVICE)   # (1,T)
        mask_in   = torch.from_numpy(mask_in_np[None, ...]).to(DEVICE)     # (1,T)
        mask_oot  = torch.from_numpy(mask_oot_np[None, ...]).to(DEVICE)    # (1,T)
        oot_mean  = torch.from_numpy(oot_mean_np[None, ...]).to(DEVICE)    # (1,L)
        oot_std   = torch.from_numpy(oot_std_np[None, ...]).to(DEVICE)     # (1,L)
        X_feat    = torch.from_numpy(X_feat_np[None, ...]).to(DEVICE)      # (1,16)

        mu, sigma = model(X_spec, time_mask, mask_in, mask_oot, oot_mean, oot_std, X_feat, SIG_FLOOR)
        mu_np     = mu.squeeze(0).detach().cpu().numpy().astype(np.float32)
        sigma_np  = np.clip(sigma.squeeze(0).detach().cpu().numpy(), 1e-8, None).astype(np.float32)
        return mu_np, sigma_np

print("Running test inference and building submission...")
sample_cols = pd.read_csv(DATA_DIR / "sample_submission.csv", nrows=0).columns.tolist()
cols_mu    = [c for c in sample_cols if c.startswith("wl_")]
cols_sigma = [c for c in sample_cols if c.startswith("sigma_")]
assert len(cols_mu) == 283 and len(cols_sigma) == 283

test_star = pd.read_csv(DATA_DIR / "test_star_info.csv").set_index("planet_id").sort_index()
test_pids = test_star.index.astype(np.int64).tolist()

results_per_planet = {}
for pid in test_pids:
    # Metadata features standardized with train statistics.
    star_vals = test_star.loc[pid].values.astype(np.float32)  # (8,)
    feat16    = planet_features16(star_vals)                  # (16,)
    feat16z   = standardize_apply(feat16, feat_mu, feat_sd)   # (16,)

    # Preprocess all usable test observations for this planet.
    try:
        obs_dict = preprocess_planet_obs(DATA_DIR, "test", pid)  # {obs_idx: {"M","p1","p2"}}
    except Exception as exc:
        print(f"[WARN] Failed to preprocess test planet {pid}: {type(exc).__name__}: {exc}")
        obs_dict = {}

    mu_list, sigma_list = [], []
    if len(obs_dict) == 0:
        # Defensive fallback if no usable observation is available.
        X_spec    = np.zeros((T_TARGET, 283), dtype=np.float32)
        time_mask = np.zeros((T_TARGET,), dtype=bool); time_mask[0] = True
        mask_in   = np.zeros((T_TARGET,), dtype=bool)
        mask_oot  = time_mask.copy()
        oot_mean  = np.zeros((283,), dtype=np.float32)
        oot_std   = np.ones ((283,), dtype=np.float32)

        mu_np, sigma_np = infer_one_observation_specmix(
            X_spec, time_mask, mask_in, mask_oot, oot_mean, oot_std, feat16z
        )
        mu_list.append(mu_np); sigma_list.append(sigma_np)
    else:
        for obs_idx, pack in sorted(obs_dict.items()):
            M  = pack["M"]; p1 = int(pack["p1"]); p2 = int(pack["p2"])

            # Fixed-length time axis and validity mask.
            X_spec, time_mask = pad_or_crop_time(M, T_TARGET)

            # Transit masks after the same crop.
            mask_in, mask_oot = build_masks_after_crop(M.shape[0], T_TARGET, p1, p2, DELTA_FRAMES)

            # Per-channel out-of-transit baseline.
            valid_oot = (mask_oot & time_mask)
            if valid_oot.any():
                oot_mean = np.nanmean(X_spec[valid_oot], axis=0)
                oot_std  = np.nanstd( X_spec[valid_oot], axis=0, ddof=0)
            else:
                valid = time_mask
                oot_mean = np.nanmean(X_spec[valid], axis=0)
                oot_std  = np.nanstd( X_spec[valid], axis=0, ddof=0)
            
            # Replace invalid values and keep a strictly positive scale.
            oot_mean = np.nan_to_num(oot_mean, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
            oot_std  = np.nan_to_num(oot_std,  nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
            oot_std  = np.maximum(oot_std, 1e-6)

            # Model prediction.
            mu_np, sigma_np = infer_one_observation_specmix(
                X_spec, time_mask, mask_in, mask_oot, oot_mean, oot_std, feat16z
            )
            mu_list.append(mu_np); sigma_list.append(sigma_np)

    mu_agg, sigma_agg = _aggregate_observations(mu_list, sigma_list)
    results_per_planet[pid] = (mu_agg, sigma_agg)

# Assemble submission.
sub_rows = []
for pid in test_pids:
    mu_agg, sigma_agg = results_per_planet[pid]
    sub_rows.append([int(pid)] + mu_agg.tolist() + sigma_agg.tolist())

out_path = WORK_DIR / "submission.csv"
submission = pd.DataFrame(sub_rows, columns=["planet_id"] + cols_mu + cols_sigma)
submission.to_csv(out_path, index=False)
print(f"submission.csv written: {submission.shape} -> {out_path}")



## 14. Qualitative Validation Plots

These plots compare predicted spectra against the validation targets for a small number of planets. They are intended as a sanity check on spectral shape and uncertainty scale, not as the official Kaggle metric.


In [ ]:
import matplotlib.pyplot as plt

# Single-observation inference helper.
@torch.no_grad()
def _infer_specmix_single(sample):
    model.eval()
    X_spec    = torch.from_numpy(sample['X_spec'][None, ...]).to(DEVICE)
    time_mask = torch.from_numpy(sample['time_mask'][None, ...]).to(DEVICE)
    mask_in   = torch.from_numpy(sample['mask_in'][None, ...]).to(DEVICE)
    mask_oot  = torch.from_numpy(sample['mask_oot'][None, ...]).to(DEVICE)
    oot_mean  = torch.from_numpy(sample['oot_mean'][None, ...]).to(DEVICE)
    oot_std   = torch.from_numpy(sample['oot_std'][None, ...]).to(DEVICE)
    X_feat    = torch.from_numpy(sample['X_feat'][None, ...]).to(DEVICE)

    mu, sigma = model(X_spec, time_mask, mask_in, mask_oot, oot_mean, oot_std, X_feat, SIG_FLOOR)
    mu_np     = mu.squeeze(0).cpu().numpy().astype(np.float32)
    sigma_np  = np.clip(sigma.squeeze(0).cpu().numpy(), 1e-8, None).astype(np.float32)
    return mu_np, sigma_np

# Pick two validation planets.
val_planet_ids = sorted(set(int(train_samples[i]['meta']['planet_id']) for i in idx_va))
viz_rng = random.Random(SEED)
chosen_pids = viz_rng.sample(val_planet_ids, k=min(2, len(val_planet_ids)))

for pid in chosen_pids:
    # Validation observations for this planet.
    obs_idxs = [i for i in idx_va if int(train_samples[i]['meta']['planet_id']) == pid]
    mu_list, sg_list = [], []
    for i in obs_idxs:
        s = train_samples[i]
        mu_np, sg_np = _infer_specmix_single(s)
        mu_list.append(mu_np); sg_list.append(sg_np)

    mu_agg, sg_agg = _aggregate_observations(mu_list, sg_list)
    y_true = train_samples[obs_idxs[0]]['y']

    x = LAM if 'LAM' in globals() else np.arange(len(mu_agg))
    x_label = "wavelength (µm)" if 'LAM' in globals() else "spectral index"

    # Shared-axis comparison.
    plt.figure(figsize=(12, 5))
    plt.plot(x, y_true, label="ground truth (y)", linewidth=2, color="gold")
    plt.plot(x, mu_agg, label="prediction (μ̂)", linewidth=2)
    plt.fill_between(x, mu_agg - sg_agg, mu_agg + sg_agg, alpha=0.25, label="±σ̂")
    plt.title(f"Planet {pid} - ground truth vs prediction (shared y-axis)")
    plt.xlabel(x_label); plt.ylabel("Amplitude"); plt.grid(alpha=0.25); plt.legend(); plt.tight_layout()
    plt.show()

    # Dual-axis comparison for scale differences.
    fig, ax1 = plt.subplots(figsize=(12, 5))
    l1, = ax1.plot(x, y_true, linewidth=2, color="gold", label="ground truth")
    ax1.set_xlabel(x_label); ax1.set_ylabel("Amplitude (ground truth)", color="gold")
    ax1.tick_params(axis='y', labelcolor="gold"); ax1.grid(alpha=0.25)

    ax2 = ax1.twinx()
    l2, = ax2.plot(x, mu_agg, linewidth=2, label="prediction (μ̂)")
    ax2.set_ylabel("Amplitude (predicted)", color=l2.get_color())
    ax2.tick_params(axis='y', labelcolor=l2.get_color())

    lines = [l1, l2]; labels = [ln.get_label() for ln in lines]
    ax1.legend(lines, labels, loc="best")
    plt.title(f"Planet {pid} - ground truth (left) vs prediction (right)")
    plt.tight_layout(); plt.show()
